# Construção da Base de Conhecimento

Este notebook gera os embeddings das avaliações e constrói o índice FAISS que será utilizado pelo pipeline RAG.

**Pré-requisito:** execute `01_exploracao.ipynb` primeiro para gerar `data/processed/documents.json`.

**Saídas geradas:**
- `data/processed/reviews.index` — índice FAISS
- `data/processed/documents.pkl` — lista de documentos correspondente aos vetores

In [1]:
import sys
from pathlib import Path

# Permite importar src/ a partir de notebooks/
sys.path.insert(0, str(Path().resolve().parent))

DATA_PROCESSED = Path("../data/processed")
DOCUMENTS_PATH = DATA_PROCESSED / "documents.json"

assert DOCUMENTS_PATH.exists(), (
    f"Arquivo não encontrado: {DOCUMENTS_PATH}\n"
    "Execute 01_exploracao.ipynb antes de continuar."
)

## 1. Carregando os documentos

In [2]:
from src.knowledge_base import load_documents

documents = load_documents(DOCUMENTS_PATH)
print(f"Total de documentos: {len(documents):,}")
print()
print("Exemplo de documento:")
doc = documents[0]
print(f"  Texto    : {doc['text'][:120]}...")
print(f"  Metadados: {doc['metadata']}")

W1106 11:02:51.340000 79383 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Total de documentos: 42,687

Exemplo de documento:
  Texto    : Recebi bem antes do prazo estipulado....
  Metadados: {'review_id': 'e64fb393e7b32834bb789ff8bb30750e', 'order_id': '658677c97b385a9be170737859d3511b', 'review_score': 5, 'review_creation_date': '2017-04-21T00:00:00', 'purchase_year': 2017, 'purchase_month': 4, 'delivery_delta_days': -20.0, 'product_category_name_english': 'garden_tools'}


## 2. Modelo de embeddings — decisão técnica

Utilizamos o modelo **`paraphrase-multilingual-MiniLM-L12-v2`** da biblioteca `sentence-transformers`.

| Critério | Justificativa |
|---|---|
| **Suporte a português** | Modelo treinado em 50+ idiomas, incluindo português. Captura semântica de termos como 'entrega atrasada', 'produto com defeito', 'ótimo atendimento' sem necessidade de tradução. |
| **Leveza** | 118 M parâmetros (~420 MB). Roda em CPU sem GPU, viabilizando uso local e em cloud gratuita. |
| **Dimensão dos vetores** | 384 dimensões — equilíbrio entre expressividade e custo de armazenamento/busca. |
| **Similaridade semântica** | Otimizado para *paraphrase similarity*, que se alinha bem ao objetivo de recuperar avaliações com significado próximo à pergunta do usuário. |

In [3]:
from sentence_transformers import SentenceTransformer
from src.knowledge_base import EMBEDDING_MODEL

print(f"Carregando modelo: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)

# Teste rápido de codificação
test_embedding = model.encode(["produto chegou com defeito"])
print(f"Dimensão do vetor: {test_embedding.shape[1]}")

Carregando modelo: paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimensão do vetor: 384


## 3. Estratégia de indexação — decisão técnica

Utilizamos **FAISS `IndexFlatIP`** (inner product) com vetores **normalizados em L2**.

Produto interno entre vetores unitários = **similaridade cosseno**. Isso garante que a relevância reflita a proximidade semântica e não o comprimento do texto.

**Por que FAISS?**
- Sem servidor externo — o índice é um arquivo em disco
- Determinístico e auditável — `IndexFlatIP` faz busca exata (sem aproximação)
- Escala bem para os ~40 mil documentos do dataset Olist

## 4. Gerando os embeddings e construindo o índice

In [4]:
from src.knowledge_base import build_index

index, docs = build_index(
    documents_path=DOCUMENTS_PATH,
    output_dir=DATA_PROCESSED,
)

print(f"\nÍndice construído com {index.ntotal:,} vetores.")

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 42,687 documents…


Batches:   0%|          | 0/167 [00:00<?, ?it/s]

Index saved  → ../data/processed/reviews.index  (42,687 vectors, dim=384)
Docs saved   → ../data/processed/documents.pkl

Índice construído com 42,687 vetores.


## 5. Verificação do índice

Testamos a busca com algumas queries representativas para confirmar que as avaliações recuperadas são semanticamente coerentes com a pergunta.

In [5]:
from src.retrieval import ReviewSearcher

searcher = ReviewSearcher(DATA_PROCESSED)

test_queries = [
    "produto chegou com defeito",
    "entrega atrasada",
    "ótimo atendimento, produto excelente",
]

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print('='*70)
    results = searcher.search(query, k=3)
    if not results:
        print("  (nenhum resultado acima do limiar)")
    for i, r in enumerate(results, 1):
        meta = r.get('metadata', {})
        print(f"  [{i}] Nota: {meta.get('review_score','?')} | Sim: {r['similarity_score']:.3f}")
        print(f"      {r['text'][:150]}..." if len(r['text']) > 150 else f"      {r['text']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Query: produto chegou com defeito
  [1] Nota: 1 | Sim: 0.993
      produto veio com defeito
  [2] Nota: 1 | Sim: 0.986
      O produto veio com defeito
  [3] Nota: 1 | Sim: 0.986
      O produto veio com defeito

Query: entrega atrasada


  [1] Nota: 5 | Sim: 0.966
      A entrega atrasou
  [2] Nota: 1 | Sim: 0.910
      A entrega esta atrasada.
  [3] Nota: 1 | Sim: 0.878
      demora na entrega

Query: ótimo atendimento, produto excelente
  [1] Nota: 3 | Sim: 0.970
      ótimo atendimento e produto.
  [2] Nota: 5 | Sim: 0.953
      ótimo serviço e ótimo produto muito satisfeita!
  [3] Nota: 5 | Sim: 0.948
      Ótimo produto, ótimo atendimento!


## Resumo

| Etapa | Resultado |
|---|---|
| Documentos indexados | (ver saída acima) |
| Dimensão dos vetores | 384 |
| Modelo | paraphrase-multilingual-MiniLM-L12-v2 |
| Índice | FAISS IndexFlatIP (similaridade cosseno) |
| Arquivos gerados | `data/processed/reviews.index`, `data/processed/documents.pkl` |

**Próxima etapa:** `03_pipeline_rag.ipynb` — recuperação semântica + geração de respostas fundamentadas em evidências.